In [12]:
from dotenv import load_dotenv
import os
import json

In [2]:
load_dotenv()

True

In [3]:
os.getenv("OPENAI_API_KEY")[:50]

'sk-proj-MihNDOEbdvVMA9YPJ_LVCGatlca_fW01Ep1giqNEaP'

In [4]:
from openai import OpenAI

client = OpenAI()                      # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"


In [7]:
SYSTEM = (
    "You are a travel assistant. Use the tools for weather, flights and currency "
    "instead of guessing. When you have everything you need, answer briefly."
)

MAX_STEPS = 5

In [9]:
"""Mock travel tools shared by the labs.

Real agents call real services (weather APIs, airline systems, payment
gateways). These functions return fixed, made-up data instead, so every lab
runs offline, costs nothing beyond the model calls, and behaves the same way
every time. Only the model's wording varies between runs.
"""

import uuid

WEATHER = {
    "goa": {"temp_c": 31, "condition": "sunny", "humidity_pct": 70},
    "mumbai": {"temp_c": 29, "condition": "humid and cloudy", "humidity_pct": 82},
    "delhi": {"temp_c": 24, "condition": "hazy", "humidity_pct": 45},
    "jaipur": {"temp_c": 27, "condition": "clear", "humidity_pct": 30},
    "manali": {"temp_c": 9, "condition": "cold and clear", "humidity_pct": 40},
}

FLIGHTS = [  # fictional flights, prices in Indian rupees (INR)
    {"id": "FL-101", "origin": "Mumbai", "destination": "Goa", "departs": "07:10", "price_inr": 4200},
    {"id": "FL-202", "origin": "Mumbai", "destination": "Goa", "departs": "13:45", "price_inr": 3500},
    {"id": "FL-303", "origin": "Delhi", "destination": "Goa", "departs": "09:30", "price_inr": 6100},
    {"id": "FL-404", "origin": "Delhi", "destination": "Jaipur", "departs": "18:00", "price_inr": 2900},
    {"id": "FL-505", "origin": "Mumbai", "destination": "Jaipur", "departs": "11:15", "price_inr": 5200},
]

RATES_TO_INR = {"INR": 1.0, "USD": 88.0, "EUR": 103.0, "GBP": 118.0}  # fixed demo rates, not live


def get_weather(city: str) -> dict:
    """Get the current weather for a city.

    Args:
        city: City name, for example "Goa".
    """
    data = WEATHER.get(city.strip().lower())
    if data is None:
        return {"error": f"No weather data for {city}. Known cities: {', '.join(WEATHER)}."}
    return {"city": city, **data}


def search_flights(origin: str, destination: str) -> dict:
    """Find flights between two cities, cheapest first.

    Args:
        origin: Departure city, for example "Mumbai".
        destination: Arrival city, for example "Goa".
    """
    matches = [f for f in FLIGHTS
               if f["origin"].lower() == origin.strip().lower()
               and f["destination"].lower() == destination.strip().lower()]
    return {"flights": sorted(matches, key=lambda f: f["price_inr"])}


def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount between INR, USD, EUR and GBP.

    Args:
        amount: The amount to convert.
        from_currency: Currency code to convert from, for example "USD".
        to_currency: Currency code to convert to, for example "INR".
    """
    src, dst = from_currency.upper(), to_currency.upper()
    if src not in RATES_TO_INR or dst not in RATES_TO_INR:
        return {"error": f"Supported currencies: {', '.join(RATES_TO_INR)}."}
    converted = amount * RATES_TO_INR[src] / RATES_TO_INR[dst]
    return {"amount": amount, "from": src, "to": dst, "result": round(converted, 2)}


def book_flight(flight_id: str, passenger: str) -> dict:
    """Book a seat on a flight. This action spends money and cannot be undone.

    Args:
        flight_id: The flight id returned by search_flights, for example "FL-202".
        passenger: Full name of the passenger.
    """
    flight = next((f for f in FLIGHTS if f["id"] == flight_id), None)
    if flight is None:
        return {"error": f"Unknown flight id {flight_id}."}
    return {"confirmation": "BK-" + uuid.uuid4().hex[:6].upper(), "passenger": passenger, **flight}


# The same tools described for the model, in the JSON Schema format that the
# Chat Completions API expects. The model only ever sees these descriptions.
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {"type": "object",
                       "properties": {"city": {"type": "string", "description": "City name, e.g. Goa"}},
                       "required": ["city"]}}},



    
    {"type": "function", "function": {
        "name": "search_flights",
        "description": "Find flights between two cities, cheapest first. Prices are in INR.",
        "parameters": {"type": "object",
                       "properties": {"origin": {"type": "string"}, "destination": {"type": "string"}},
                       "required": ["origin", "destination"]}}},


    
    {"type": "function", "function": {
        "name": "convert_currency",
        "description": "Convert an amount between INR, USD, EUR and GBP.",
        "parameters": {"type": "object",
                       "properties": {"amount": {"type": "number"},
                                      "from_currency": {"type": "string"},
                                      "to_currency": {"type": "string"}},
                       "required": ["amount", "from_currency", "to_currency"]}}},
]

# Name -> Python function: how our code finds the function the model asked for.
TOOLS = {"get_weather": get_weather, "search_flights": search_flights, "convert_currency": convert_currency}

In [31]:
def run_tool(name: str, arguments: str, tool_functions: dict) -> dict:
    """Runs one requested tool. Errors are returned as data the model can read and react to."""
    if name not in tool_functions:
        return {"error": f"Unknown tool {name}."}
    try:
        return tool_functions[name](**json.loads(arguments))
    except Exception as exc:  # bad arguments, a failing service, ...
        return {"error": f"{type(exc).__name__}: {exc}"}


def run_agent_loop(messages: list, tool_schemas: list, tool_functions: dict,
                   max_steps: int = MAX_STEPS, verbose: bool = True) -> tuple[str, list]:
    trace = []

    for step in range(1, max_steps + 1):
        # 1. THINK: the model sees the whole conversation and the tool descriptions.
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tool_schemas)
        message = response.choices[0].message
        # print(f"message in step {step}:: {message}")

        # 2. DONE? No tool requested means the model has produced its final answer.
        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            return message.content, trace


        # 3. ACT: record the model's tool requests, then run each one.
        messages.append({"role": "assistant", "content": message.content, "tool_calls": [
            {"id": call.id, "type": "function",
             "function": {"name": call.function.name, "arguments": call.function.arguments}}
            for call in message.tool_calls]})

        for call in message.tool_calls:
            result = run_tool(call.function.name, call.function.arguments, tool_functions)
            trace.append({"step": step, "tool": call.function.name,
                          "arguments": call.function.arguments, "result": result})

            if verbose:
                print(f"  step {step}: {call.function.name}({call.function.arguments}) -> {result}")
        
    
            # 4. OBSERVE: the result goes back into the conversation for the next iteration.
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
        
    return "I stopped because I reached the step limit before finishing.", trace
    
def run_agent(user_request: str) -> str:
    """Convenience wrapper: one request in, one answer out."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_request}]
    answer, trace = run_agent_loop(messages, TOOL_SCHEMAS, TOOLS)
    return answer, trace

    

In [32]:
response = run_agent(
        "I'm flying from Mumbai to Goa this weekend with 500 US dollars. What's the weather there, "
        "which flight is cheapest, and how many rupees will I have left after paying for it?"
    )

  step 1: get_weather({"city": "Goa"}) -> {'city': 'Goa', 'temp_c': 31, 'condition': 'sunny', 'humidity_pct': 70}
  step 1: search_flights({"origin": "Mumbai", "destination": "Goa"}) -> {'flights': [{'id': 'FL-202', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '13:45', 'price_inr': 3500}, {'id': 'FL-101', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '07:10', 'price_inr': 4200}]}
  step 1: convert_currency({"amount": 500, "from_currency": "USD", "to_currency": "INR"}) -> {'amount': 500, 'from': 'USD', 'to': 'INR', 'result': 44000.0}


In [33]:
print(response)

("The weather in Goa is sunny with a temperature of 31°C and a humidity of 70%. \n\nThe cheapest flight from Mumbai to Goa costs ₹3,500, departing at 13:45. \n\nConverting your $500 gives you ₹44,000. After paying for the flight, you'll have ₹40,500 left.", [{'step': 1, 'tool': 'get_weather', 'arguments': '{"city": "Goa"}', 'result': {'city': 'Goa', 'temp_c': 31, 'condition': 'sunny', 'humidity_pct': 70}}, {'step': 1, 'tool': 'search_flights', 'arguments': '{"origin": "Mumbai", "destination": "Goa"}', 'result': {'flights': [{'id': 'FL-202', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '13:45', 'price_inr': 3500}, {'id': 'FL-101', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '07:10', 'price_inr': 4200}]}}, {'step': 1, 'tool': 'convert_currency', 'arguments': '{"amount": 500, "from_currency": "USD", "to_currency": "INR"}', 'result': {'amount': 500, 'from': 'USD', 'to': 'INR', 'result': 44000.0}}])


In [34]:
response = run_agent(
        "I'm flying from Mumbai to Goa this weekend with 500 US dollars. What's the weather there, "
        "which flight is cheapest, and how many dollars will I have left after paying for it?"
    )

  step 1: get_weather({"city": "Goa"}) -> {'city': 'Goa', 'temp_c': 31, 'condition': 'sunny', 'humidity_pct': 70}
  step 1: search_flights({"origin": "Mumbai", "destination": "Goa"}) -> {'flights': [{'id': 'FL-202', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '13:45', 'price_inr': 3500}, {'id': 'FL-101', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '07:10', 'price_inr': 4200}]}
  step 1: convert_currency({"amount": 500, "from_currency": "USD", "to_currency": "INR"}) -> {'amount': 500, 'from': 'USD', 'to': 'INR', 'result': 44000.0}
  step 2: convert_currency({"amount":3500,"from_currency":"INR","to_currency":"USD"}) -> {'amount': 3500, 'from': 'INR', 'to': 'USD', 'result': 39.77}


In [35]:
print(response)

("The weather in Goa this weekend is sunny with a temperature of 31°C and 70% humidity. The cheapest flight from Mumbai to Goa costs ₹3,500 (approximately $39.77). \n\nAfter paying for the flight, you'll have about $460.23 left from your initial $500.", [{'step': 1, 'tool': 'get_weather', 'arguments': '{"city": "Goa"}', 'result': {'city': 'Goa', 'temp_c': 31, 'condition': 'sunny', 'humidity_pct': 70}}, {'step': 1, 'tool': 'search_flights', 'arguments': '{"origin": "Mumbai", "destination": "Goa"}', 'result': {'flights': [{'id': 'FL-202', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '13:45', 'price_inr': 3500}, {'id': 'FL-101', 'origin': 'Mumbai', 'destination': 'Goa', 'departs': '07:10', 'price_inr': 4200}]}}, {'step': 1, 'tool': 'convert_currency', 'arguments': '{"amount": 500, "from_currency": "USD", "to_currency": "INR"}', 'result': {'amount': 500, 'from': 'USD', 'to': 'INR', 'result': 44000.0}}, {'step': 2, 'tool': 'convert_currency', 'arguments': '{"amount":3500,"from_curre

In [37]:
response[1]

[{'step': 1,
  'tool': 'get_weather',
  'arguments': '{"city": "Goa"}',
  'result': {'city': 'Goa',
   'temp_c': 31,
   'condition': 'sunny',
   'humidity_pct': 70}},
 {'step': 1,
  'tool': 'search_flights',
  'arguments': '{"origin": "Mumbai", "destination": "Goa"}',
  'result': {'flights': [{'id': 'FL-202',
     'origin': 'Mumbai',
     'destination': 'Goa',
     'departs': '13:45',
     'price_inr': 3500},
    {'id': 'FL-101',
     'origin': 'Mumbai',
     'destination': 'Goa',
     'departs': '07:10',
     'price_inr': 4200}]}},
 {'step': 1,
  'tool': 'convert_currency',
  'arguments': '{"amount": 500, "from_currency": "USD", "to_currency": "INR"}',
  'result': {'amount': 500, 'from': 'USD', 'to': 'INR', 'result': 44000.0}},
 {'step': 2,
  'tool': 'convert_currency',
  'arguments': '{"amount":3500,"from_currency":"INR","to_currency":"USD"}',
  'result': {'amount': 3500, 'from': 'INR', 'to': 'USD', 'result': 39.77}}]